# QuantJourney SDK - Crypto Perps Basis and Funding Arbitrage

This notebook demonstrates a QuantJourney SDK workflow that pulls CCXT-style spot/perp data and funding history where available, then builds funding and basis diagnostics.

It covers:

- Direct QuantJourney SDK calls for the required market, macro, regulatory or portfolio data
- Transparent pandas/numpy calculations so research assumptions stay visible
- Chart-ready output that can be reused in notebooks, reports or API documentation

## Prerequisites

Make sure you have:

- Access to QuantJourney API (https://api.quantjourney.cloud)
- `QJ_API_KEY` configured in your environment
- Tenant access to the connectors used by this example

## Imports and Plot Style

In [ ]:
import os
import math
import json
from pathlib import Path
from typing import Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from quantjourney.sdk import QuantJourneyAPI
plt.style.use('default')
plt.rcParams.update({'figure.figsize': (12, 6), 'axes.grid': True, 'grid.alpha': 0.25, 'axes.spines.top': False, 'axes.spines.right': False})


## QuantJourney Client

In [ ]:
qj = QuantJourneyAPI(api_key=os.environ['QJ_API_KEY'])
START = os.getenv('QJ_EXAMPLE_START', '2020-01-01')
END = os.getenv('QJ_EXAMPLE_END') or pd.Timestamp.today().normalize().strftime('%Y-%m-%d')


## Response Helpers

In [ ]:
def unwrap(payload: Any) -> Any:
    """Return the useful data value from common QuantJourney response shapes."""
    if isinstance(payload, dict) and 'data' in payload:
        payload = payload['data']
    if isinstance(payload, dict) and 'value' in payload:
        return payload['value']
    return payload

def as_rows(payload: Any) -> list[dict[str, Any]]:
    value = unwrap(payload)
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        for key in ('rows', 'data', 'items', 'prices', 'results'):
            if isinstance(value.get(key), list):
                return value[key]
        return [value]
    return []


## Portfolio and Risk Helpers

In [ ]:
def max_drawdown(nav: pd.Series) -> float:
    drawdown = nav / nav.cummax() - 1
    return float(drawdown.min())

def performance_stats(ret: pd.Series) -> pd.Series:
    ret = ret.dropna()
    nav = (1 + ret).cumprod()
    ann_ret = nav.iloc[-1] ** (252 / len(ret)) - 1 if len(ret) and nav.iloc[-1] > 0 else np.nan
    ann_vol = ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol and np.isfinite(ann_vol) else np.nan
    return pd.Series({'annual_return': ann_ret, 'annual_volatility': ann_vol, 'sharpe': sharpe, 'max_drawdown': max_drawdown(nav) if len(nav) else np.nan, 'total_return': nav.iloc[-1] - 1 if len(nav) else np.nan})

def inverse_vol_weights(ret: pd.DataFrame) -> pd.Series:
    vol = ret.std().replace(0, np.nan)
    inv = 1 / vol
    return (inv / inv.sum()).fillna(0)

def min_variance_weights(ret: pd.DataFrame, ridge: float=0.0001) -> pd.Series:
    cov = ret.cov().fillna(0).to_numpy() * 252
    cov = cov + np.eye(cov.shape[0]) * ridge
    inv = np.linalg.pinv(cov)
    raw = inv @ np.ones(cov.shape[0])
    raw = np.maximum(raw, 0)
    if raw.sum() == 0:
        raw = np.ones(cov.shape[0])
    return pd.Series(raw / raw.sum(), index=ret.columns)

def portfolio_returns(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    return aligned @ weights.reindex(aligned.columns).fillna(0)

def risk_contribution(ret: pd.DataFrame, weights: pd.Series) -> pd.Series:
    aligned = ret[weights.index].dropna()
    cov = aligned.cov() * 252
    w = weights.reindex(cov.columns).fillna(0).to_numpy()
    port_var = float(w @ cov.to_numpy() @ w)
    if port_var <= 0:
        return pd.Series(0.0, index=cov.columns)
    contrib = w * (cov.to_numpy() @ w) / port_var
    return pd.Series(contrib, index=cov.columns)


In [ ]:
btc_spot = qj.ccxt.get_historical_prices(symbol='BTC/USDT', exchange='binance', timeframe='1d', since=START)
funding = qj.ccxt.get_historical_funding_rates(symbol='BTC/USDT', exchange='binance')
open_interest = qj.ccxt.get_open_interest(symbol='BTC/USDT', exchange='binance')
gecko = qj.coingecko.get_historical_prices(coin_id='bitcoin', vs_currency='usd', days='max')


In [ ]:
rows = as_rows(btc_spot) or as_rows(gecko)
prices_df = pd.DataFrame(rows)
if not prices_df.empty:
    date_col = 'date' if 'date' in prices_df else prices_df.columns[0]
    numeric_cols = prices_df.select_dtypes(include='number').columns
    value_col = 'close' if 'close' in prices_df else numeric_cols[-1] if len(numeric_cols) else prices_df.columns[-1]
    prices_df['date'] = pd.to_datetime(prices_df[date_col], errors='coerce')
    prices_df['price'] = pd.to_numeric(prices_df[value_col], errors='coerce')
    prices_df = prices_df.dropna(subset=['date', 'price']).set_index('date').sort_index()
    btc_ret = prices_df['price'].pct_change().dropna()
else:
    btc_ret = pd.Series(dtype=float)
funding_df = pd.DataFrame(as_rows(funding))
display(funding_df.head())
if not btc_ret.empty:
    nav = (1 + btc_ret).cumprod()
    nav.plot(title='BTC spot context for funding/basis research')
    plt.show()
    display(performance_stats(btc_ret))


## Notes

This is an example workflow. In production, tenant scopes, connector allowlists,
provider metadata, request IDs and audit logs should be retained next to the resulting
tables or charts.